# Mould Design & Manufacturability (DFM) Calculator
**Plastic Design Calculators — Notebook 5**

---

## Part 1: Theory and Governing Equations

### 1.1 Why DFM Matters

A structurally optimal plastic design is useless if it cannot be injection-moulded. The process imposes hard geometric constraints: parts must shrink predictably, eject without damage, and fill without voids. This notebook automates the core Design-for-Manufacturability (DFM) checks.

### 1.2 Shrinkage Compensation

As the polymer melt cools from injection temperature to ambient, it contracts according to its pressure-volume-temperature (pvT) relationship. The steel mould cavity must be deliberately oversized:

$$\boxed{M_d = \frac{P_d}{1 - S_r}}$$

where:
- $M_d$ — required mould cavity dimension [mm]
- $P_d$ — desired finished part dimension [mm]
- $S_r$ — volumetric shrinkage rate (e.g. $0.015$ for 1.5%)

### 1.3 Draft Angle Rules

All surfaces parallel to the direction of mould opening must have a taper (draft angle) to allow ejection without friction damage:

| Surface Type | Minimum Draft |
|---|---|
| Smooth external surface | 0.5° absolute minimum |
| Standard vertical wall | 1° per 25.4 mm (1 inch) depth |
| Textured surface | 1° per 0.025 mm texture depth |

> **Critical Failure:** Any 0° draft surface immediately triggers a manufacturing FAIL.

### 1.4 Wall Thickness Uniformity

Non-uniform walls create differential cooling rates, leading to warpage, sink marks, and residual stress. The uniformity check:

$$\text{Variation} = \frac{t_{\max} - t_{\min}}{t_{\max}} \leq 10\%$$

### 1.5 Gate Positioning Rule

Molten polymer must enter at the **thickest cross-section** to ensure adequate packing pressure before freeze-off in thin areas.

### Assumptions
- Single-cavity mould (no family tooling compensation)
- Isotropic shrinkage (valid for amorphous; semi-crystalline may have anisotropic shrinkage)
- No weld line, flow balance, or cooling circuit analysis included
- Gate position evaluated qualitatively (coordinate input required)

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import SHRINKAGE_RATES

# ── Material ──────────────────────────────────────────────────────────────────
MATERIAL = 'PP'
S_r      = SHRINKAGE_RATES[MATERIAL]

# ── Part dimensions to compensate ────────────────────────────────────────────
# List of (dimension_name, desired_part_dim_mm)
part_dimensions = [
    ('Length',  Q_(150.0, 'mm')),
    ('Width',   Q_( 60.0, 'mm')),
    ('Height',  Q_( 30.0, 'mm')),
    ('Boss OD', Q_(  8.0, 'mm')),
]

# ── Draft angles ──────────────────────────────────────────────────────────────
# List of (surface_name, draft_angle_deg, depth_mm)
draft_surfaces = [
    ('Side wall A',   1.5, Q_(30.0, 'mm')),
    ('Rib face B',    1.0, Q_(20.0, 'mm')),
    ('Boss exterior', 0.5, Q_(10.0, 'mm')),
    ('Textured lid',  0.0, Q_( 5.0, 'mm')),   # intentional FAIL for demonstration
]

# ── Wall thickness profile ─────────────────────────────────────────────────────
# Array of measured wall thicknesses across the part cross-section [mm]
wall_thicknesses_mm = np.array([2.0, 2.1, 2.0, 1.9, 2.2, 2.0, 3.5, 2.1, 2.0, 1.95])
# Note: 3.5 mm entry is a deliberate heavy section to trigger warning

# ── Gate position ─────────────────────────────────────────────────────────────
# Coordinate [mm] of proposed gate location within thickness profile array index
gate_thickness_idx = 6   # index into wall_thicknesses_mm

print(f"Material     : {MATERIAL}")
print(f"Shrinkage Sr : {S_r*100:.1f} %")

---
## Part 3: Computation Engine

In [ ]:
# ── 3.1  Symbolic equations ───────────────────────────────────────────────────
Pd_s, Sr_s, Md_s = sp.symbols('P_d S_r M_d', positive=True)
t_max_s, t_min_s = sp.symbols('t_max t_min', positive=True)

shrinkage_eq  = Pd_s / (1 - Sr_s)
variation_eq  = (t_max_s - t_min_s) / t_max_s

print("Mould dimension M_d =")
sp.pprint(shrinkage_eq)
print()
print("Thickness variation =")
sp.pprint(variation_eq)

In [ ]:
def mould_dimension(part_dim_m, shrinkage_rate, **kwargs):
    """Required mould cavity dimension compensating for polymer shrinkage.

    Args:
        part_dim_m (float | numpy.ndarray): Target finished part dimension [m].
        shrinkage_rate (float): Volumetric shrinkage rate (e.g. 0.015 for 1.5%).
        **kwargs: Reserved for anisotropic shrinkage tensors (semi-crystalline).

    Returns:
        float | numpy.ndarray: Required mould cavity dimension [m].
    """
    return part_dim_m / (1.0 - shrinkage_rate)


def check_draft_angle(draft_deg, depth_mm, texture_depth_mm=0.0, **kwargs):
    """Validate draft angle against DFM rules.

    Args:
        draft_deg (float): Actual draft angle on the surface [degrees].
        depth_mm (float): Cavity depth for this surface [mm].
        texture_depth_mm (float): Texture amplitude if surface is textured [mm].
        **kwargs: Reserved for EDM spark erosion roughness offsets.

    Returns:
        dict: {'pass': bool, 'required_deg': float, 'actual_deg': float}
    """
    if draft_deg == 0.0:
        return {'pass': False, 'required_deg': 0.5, 'actual_deg': 0.0,
                'reason': 'Zero draft — absolute failure'}

    # Standard rule: 1° per 25.4 mm depth
    required_standard = depth_mm / 25.4
    # Absolute minimum
    required_absolute = 0.5
    # Texture requirement
    required_texture  = texture_depth_mm / 0.025 if texture_depth_mm > 0 else 0.0

    required = max(required_standard, required_absolute, required_texture)
    passed   = draft_deg >= required
    return {'pass': passed, 'required_deg': required, 'actual_deg': draft_deg,
            'reason': 'OK' if passed else f'Need ≥ {required:.2f}°'}


def wall_thickness_check(thicknesses_mm, tolerance=0.10, **kwargs):
    """Check wall thickness uniformity across a cross-sectional profile.

    Args:
        thicknesses_mm (numpy.ndarray): Array of wall thickness measurements [mm].
        tolerance (float): Maximum allowable variation ratio (default 0.10 = 10%).
        **kwargs: Reserved for rib-to-wall ratio sub-checks.

    Returns:
        dict: variation, t_min, t_max, t_mean, passed.
    """
    t_max = np.max(thicknesses_mm)
    t_min = np.min(thicknesses_mm)
    variation = (t_max - t_min) / t_max
    return {
        'variation': variation,
        't_min_mm':  t_min,
        't_max_mm':  t_max,
        't_mean_mm': np.mean(thicknesses_mm),
        'passed':    variation <= tolerance,
    }


def gate_position_check(thicknesses_mm, gate_idx, **kwargs):
    """Verify that the gate is located at the thickest cross-section.

    Args:
        thicknesses_mm (numpy.ndarray): Wall thickness profile [mm].
        gate_idx (int): Index of the proposed gate location in the thickness array.
        **kwargs: Reserved for flow-length/thickness ratio checks.

    Returns:
        dict: passed, gate_thickness, max_thickness, recommended_idx.
    """
    max_t   = np.max(thicknesses_mm)
    gate_t  = thicknesses_mm[gate_idx]
    rec_idx = int(np.argmax(thicknesses_mm))
    return {
        'passed':          gate_t == max_t,
        'gate_thickness':  gate_t,
        'max_thickness':   max_t,
        'recommended_idx': rec_idx,
    }


print("DFM functions defined.")

In [ ]:
# ── 3.2  Numerical evaluation ─────────────────────────────────────────────────

# Shrinkage compensation (vectorised)
shrinkage_results = []
for name, dim in part_dimensions:
    pd_m = strip_units(dim.to('meter'))
    md_m = mould_dimension(pd_m, S_r)
    shrinkage_results.append({
        'name': name,
        'P_d_mm': pd_m * 1e3,
        'M_d_mm': md_m * 1e3,
        'delta_mm': (md_m - pd_m) * 1e3,
    })

# Draft angles
draft_results = []
for surface_name, angle, depth in draft_surfaces:
    d_mm = strip_units(depth.to('mm'))
    result = check_draft_angle(angle, d_mm)
    result['name'] = surface_name
    draft_results.append(result)

# Wall thickness
wall_result = wall_thickness_check(wall_thicknesses_mm)

# Gate position
gate_result = gate_position_check(wall_thicknesses_mm, gate_thickness_idx)

print("\nShrinkage Compensation:")
for r in shrinkage_results:
    print(f"  {r['name']:12s}: Pd={r['P_d_mm']:.2f} mm → Md={r['M_d_mm']:.3f} mm  (+{r['delta_mm']:.3f} mm)")

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Wall thickness profile ────────────────────────────────────────────
ax1 = axes[0]
x   = np.arange(len(wall_thicknesses_mm))
colors = ['tomato' if t > wall_result['t_mean_mm'] * 1.10 or t < wall_result['t_mean_mm'] * 0.90
          else 'steelblue' for t in wall_thicknesses_mm]
ax1.bar(x, wall_thicknesses_mm, color=colors, edgecolor='k', lw=0.5)
ax1.axhline(wall_result['t_mean_mm'], color='limegreen', ls='--', lw=1.5, label=f'Mean = {wall_result["t_mean_mm"]:.2f} mm')
ax1.axhline(wall_result['t_max_mm'],  color='tomato',    ls='--', lw=1,   label=f't_max = {wall_result["t_max_mm"]:.2f} mm')
ax1.axhline(wall_result['t_min_mm'],  color='orange',    ls='--', lw=1,   label=f't_min = {wall_result["t_min_mm"]:.2f} mm')
ax1.axvline(gate_thickness_idx, color='purple', ls=':', lw=2, label=f'Gate location (idx={gate_thickness_idx})')
red_patch   = mpatches.Patch(color='tomato',    label='Outside ±10% mean')
blue_patch  = mpatches.Patch(color='steelblue', label='Within ±10% mean')
ax1.legend(handles=[red_patch, blue_patch] + ax1.get_legend_handles_labels()[0][:-3] +
           ax1.get_legend_handles_labels()[0][-1:], fontsize=7)
ax1.set_xlabel('Cross-section Index')
ax1.set_ylabel('Wall Thickness [mm]')
ax1.set_title(f'Wall Thickness Profile\nVariation = {wall_result["variation"]*100:.1f}% (limit 10%)')
ax1.grid(True, axis='y', alpha=0.3)

# ── Plot 2: Draft angle compliance ────────────────────────────────────────────
ax2 = axes[1]
names    = [r['name']        for r in draft_results]
actual   = [r['actual_deg']  for r in draft_results]
required = [r['required_deg'] for r in draft_results]
passed   = [r['pass']        for r in draft_results]
bar_colors = ['limegreen' if p else 'tomato' for p in passed]

x2 = np.arange(len(names))
ax2.bar(x2 - 0.2, actual,   0.35, color=bar_colors, edgecolor='k', lw=0.5, label='Actual draft')
ax2.bar(x2 + 0.2, required, 0.35, color='lightgrey', edgecolor='k', lw=0.5, label='Min required')
ax2.set_xticks(x2)
ax2.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
ax2.set_ylabel('Draft Angle [degrees]')
ax2.set_title('Draft Angle DFM Check')
ax2.legend(fontsize=8)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('05_mould_design_output.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Design Rule Validation

In [ ]:
def badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

all_draft_pass = all(r['pass'] for r in draft_results)

print("═" * 68)
print("  DESIGN RULE VALIDATION — MOULD DESIGN & DFM")
print("═" * 68)

print("\n  [1] SHRINKAGE COMPENSATION")
for r in shrinkage_results:
    print(f"      {r['name']:12s}: {r['P_d_mm']:.2f} mm → mould {r['M_d_mm']:.3f} mm  (+{r['delta_mm']:.3f} mm)")

print("\n  [2] DRAFT ANGLE COMPLIANCE")
for r in draft_results:
    print(f"      {r['name']:20s}: actual={r['actual_deg']:.1f}° / min={r['required_deg']:.2f}°  {badge(r['pass'])}  {r.get('reason','')}")
print(f"      Overall draft   : {badge(all_draft_pass)}")

print("\n  [3] WALL THICKNESS UNIFORMITY")
print(f"      Variation = {wall_result['variation']*100:.1f}%  (limit 10%)")
print(f"      t_min = {wall_result['t_min_mm']:.2f} mm,  t_max = {wall_result['t_max_mm']:.2f} mm")
print(f"      Result : {badge(wall_result['passed'])}")

print("\n  [4] GATE POSITION")
print(f"      Gate at idx {gate_thickness_idx}: {gate_result['gate_thickness']:.2f} mm")
print(f"      Max thickness  : {gate_result['max_thickness']:.2f} mm  (at idx {gate_result['recommended_idx']})")
print(f"      Result : {badge(gate_result['passed'])}")
if not gate_result['passed']:
    print(f"      ⚠ Move gate to idx {gate_result['recommended_idx']} (t={gate_result['max_thickness']:.2f} mm)")

print("═" * 68)
overall = all_draft_pass and wall_result['passed'] and gate_result['passed']
if overall:
    print("  ✓ OVERALL: DESIGN PASSES all DFM criteria.")
else:
    print("  ✗ OVERALL: DESIGN FAILS DFM — see flagged items above.")
print("═" * 68)